# RAG

Для работы в рамках данного ноутбука понадобится GPU.



## 1. Векторная БД Qdrant

### 1.1. Векторный поиск

**Вектор**: `[0.1, 0.5, 0.3, ...]` — числовой массив, представляющий текст или другой объект в виде набора чисел.

Вводим понятия:
- **Dense vector**: `[0.1, 0.5, 0.3, ...]` — большинство чисел в векторе ненулевые. Embeddings - это обычно dense vectors.
- **Sparse vector**: `[0, 0, 0.3, 0, 0.5, ...]` — большинство чисел в векторе нулевые. BM25-like векторы — это обычно sparse vectors (сопоставление слово-вес).

Sparse vector хранится обычно в виде словаря, где ключ — индекс, а значение — число. Например, `[0, 0, 0.3, 0, 0.5]` может быть представлено как `{2: 0.3, 4: 0.5}`.

Векторная БД - это база данных, оптимизированная для хранения и поиска векторов. Она позволяет эффективно находить векторы, которые наиболее похожи на заданный вектор (например, по косинусной близости).

**Qdrant** - это:
- Векторная БД с открытым исходным кодом
- Поддерживает как dense, так и sparse векторы
- Чаще всего используется в системах семантического поиска и RAG (Retrieval-Augmented Generation)
- Имеет как production-версию, так и дает возможность запускать in-memory или с хранением данных на диске


In [ ]:
!pip install qdrant-client[fastembed] sentencepiece tiktoken

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

import torch

# from __config__ import *

from datasets import load_dataset

# print(len(HF_TOKEN))


# os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_HOME"] = "./hf_cache" # для колаба ок

In [ ]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:") # если хотим хранить вектора в оперативной памяти

In [ ]:
client = QdrantClient(path="qdrant_data") # если хотим хранить вектора на диске

In [ ]:
# подключение к внешнему серверу Qdrant
client = QdrantClient(
    url="http://my-qdrant-server.example.com:6333",
    api_key="YOUR_API_KEY",
    timeout=60,
)

В Qdrant данные хранятся в коллекциях (__collections__), которые состоят из точек (__points__).

Каждая точка имеет уникальный идентификатор (__id__), может содержать вектора и дополнительные данные (__payload__).

### 1.2. Полнотекстовый поиск в Qdrant

В Qdrant можно использовать BM25.

In [ ]:
from fastembed import SparseTextEmbedding

model_name = "Qdrant/bm25"

bm25_model = SparseTextEmbedding(model_name=model_name)

text = "мама мыла раму"

embedding = list(bm25_model.embed([text]))[0]

print("TEXT:")
print(text)

print("\nNON-ZERO DIMENSIONS:")
print(len(embedding.indices))

print("\nSPARSE VECTOR:")
print("indices:", embedding.indices)
print("values :", embedding.values)

In [ ]:
ds = load_dataset("json",
    data_files="https://github.com/easyise/learn_machine_learn/raw/refs/heads/master/data/gashek_rasskazy.json",
)
gashek_df = ds['train'].to_pandas()
gashek_df

In [ ]:
gashek_df = gashek_df.iloc[:-1]
gashek_df

## 2. RAG (Retrieval-Augmented Generation)

In [ ]:
%pip install -U "transformers>=4.51.0" accelerate sentencepiece safetensors

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# LLM_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
LLM_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)

llm = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

def ask_llm(question: str, max_new_tokens: int = 300) -> str:
    messages = [
        {
            "role": "system",
            "content": "Ты отвечаешь кратко и по-русски."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(llm.device)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[-1]
    answer_ids = output_ids[0][input_len:]

    return tokenizer.decode(answer_ids, skip_special_tokens=True)


questions = [
    "Что происходит в рассказе про озеро Балатон?",
    "Какой рассказ Гашека про испанскую инквизицию?",
    "В какой стране происходит действие рассказа про Швейка?"
]

for q in questions:
    print("=" * 100)
    print("QUESTION:", q)
    print()
    print(ask_llm(q))

### 2.1. Чанкинг (Chunking)

Для создания RAG необходимо выполнить chunking: разбить текст на части (чанки).

На следующем этапе мы создадим для каждого чанка эмбеддинги и сохраним их в векторной БД (например, Qdrant).

**ВОПРОС**: Почему не сохранять весь текст целиком, а разбивать его на части?

In [ ]:
def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 200) -> list[str]:
    text = str(text).replace("\r", "")
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]

    chunks = []
    current = ""

    for p in paragraphs:
        if len(current) + len(p) + 2 <= chunk_size:
            current = current + "\n\n" + p if current else p
        else:
            if current:
                chunks.append(current)

            if overlap > 0 and current:
                tail = current[-overlap:]
                current = tail + "\n\n" + p
            else:
                current = p

    if current:
        chunks.append(current)

    return chunks


rows = []

for id, story in gashek_df.iterrows():
    chunks = chunk_text(story["content"], chunk_size=1200, overlap=200)

    for chunk_id, chunk in enumerate(chunks):
        rows.append({
            "chunk_global_id": len(rows),
            "story_id": id,
            "chunk_id": chunk_id,
            "title": story["title"],
            "text": chunk,
        })

chunks_df = pd.DataFrame(rows).set_index("chunk_global_id")

print("chunks:", len(chunks_df))
chunks_df

In [ ]:
chunks_df['text'].apply(len).min()

#### Семантический чанкинг

Все ли нас устраивает в чанках фиксированного размера?

In [ ]:
!pip install -q sentence-transformers razdel

In [ ]:
from razdel import sentenize
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

embedder = SentenceTransformer("intfloat/multilingual-e5-small")

def split_ru_sentences(text: str) -> list[str]:
    sentences = [s.text.strip() for s in sentenize(str(text))]
    sentences = [s for s in sentences if len(s) > 20]
    return sentences

def semantic_split_ru(
    text: str,
    max_chunk_chars: int = 1600,
    min_chunk_chars: int = 500,
    similarity_threshold: float = 0.55,
    overlap_sentences: int = 2,
) -> list[str]:

    sentences = split_ru_sentences(text)

    if not sentences:
        return []

    if len(sentences) == 1:
        return sentences

    embeddings = embedder.encode(
        ["passage: " + s for s in sentences],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    chunks = []

    current_sentences = [sentences[0]]
    current_len = len(sentences[0])

    for i in range(1, len(sentences)):
        prev_emb = embeddings[i - 1].reshape(1, -1)
        curr_emb = embeddings[i].reshape(1, -1)

        sim = cosine_similarity(prev_emb, curr_emb)[0][0]

        sentence = sentences[i]
        sentence_len = len(sentence)

        should_split_by_semantics = (
            sim < similarity_threshold
            and current_len >= min_chunk_chars
        )

        should_split_by_length = (
            current_len + sentence_len > max_chunk_chars
        )

        if should_split_by_semantics or should_split_by_length:
            # сохраняем текущий chunk
            chunks.append(" ".join(current_sentences))

            # overlap
            overlap = current_sentences[-overlap_sentences:]

            current_sentences = overlap + [sentence]

            current_len = sum(len(s) for s in current_sentences)

        else:
            current_sentences.append(sentence)
            current_len += sentence_len

    if current_sentences:
        chunks.append(" ".join(current_sentences))

    return chunks

In [ ]:
from tqdm.auto import tqdm

rows = []

for story_id, row in tqdm(gashek_df.iterrows(), total=len(gashek_df)):
    title = row["title"]
    content = row["content"]

    chunks = semantic_split_ru(
        content,
        max_chunk_chars=450,
        min_chunk_chars=200,
        similarity_threshold=0.55,
        overlap_sentences=4,
    )

    for chunk_id, chunk in enumerate(chunks):
        rows.append({
            "chunk_global_id": len(rows),
            "story_id": story_id,
            "chunk_id": chunk_id,
            "title": title,
            "text": chunk,
            "chars": len(chunk),
        })

chunks_df = pd.DataFrame(rows).set_index("chunk_global_id")
chunks_df

In [ ]:
chunks_df['text'].apply(len).max()

### 2.2. Векторизация текста

Помещаем чанки в векторную БД - делаем то же самое, что и в гибридном поиске.

In [ ]:
# создаем новую коллекцию в Qdrant для рассказов Гашека
from qdrant_client import QdrantClient, models
from fastembed import SparseTextEmbedding
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

COLLECTION_NAME = "gashek_hybrid_rag"
DENSE_VECTOR_NAME = "dense"
SPARSE_VECTOR_NAME = "bm25"

client = QdrantClient(path="qdrant_gashek_hybrid")

dense_model = SentenceTransformer("intfloat/multilingual-e5-small")
dense_size = dense_model.get_sentence_embedding_dimension()

bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")

if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,

    vectors_config={
        DENSE_VECTOR_NAME: models.VectorParams(
            size=dense_size,
            distance=models.Distance.COSINE,
        )
    },

    sparse_vectors_config={
        SPARSE_VECTOR_NAME: models.SparseVectorParams(
            modifier=models.Modifier.IDF
        )
    },
)

print("collection created")

In [ ]:
BATCH_SIZE = 32

ids = list(chunks_df.index)
texts = chunks_df["text"].astype(str).tolist()

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="Uploading chunks"):
    end = min(start + BATCH_SIZE, len(texts))

    batch_ids = ids[start:end]
    batch_texts = texts[start:end]

    dense_vectors = dense_model.encode(
        ["passage: " + t for t in batch_texts],
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    sparse_vectors = list(bm25_model.embed(batch_texts))

    points = []

    for local_i, point_id in enumerate(batch_ids):
        row = chunks_df.loc[point_id]

        points.append(
            models.PointStruct(
                id=int(point_id),

                vector={
                    DENSE_VECTOR_NAME: dense_vectors[local_i].tolist(),

                    SPARSE_VECTOR_NAME: models.SparseVector(
                        indices=sparse_vectors[local_i].indices.tolist(),
                        values=sparse_vectors[local_i].values.tolist(),
                    ),
                },

                payload={
                    "chunk_id": int(point_id),
                    "story_id": int(row["story_id"]),
                    "local_chunk_id": int(row["chunk_id"]),
                    "title": row["title"],
                    "text": row["text"],
                },
            )
        )

    client.upsert(
        collection_name=COLLECTION_NAME,
        points=points,
    )

print("uploaded:", len(chunks_df))

### 2.3. Гибридный поиск по Чанкам

In [ ]:
def hybrid_retrieve(query: str, top_k: int = 5, prefetch_limit: int = 20) -> pd.DataFrame:
    dense_query = dense_model.encode(
        "query: " + query,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )

    sparse_query = list(bm25_model.query_embed([query]))[0]

    response = client.query_points(
        collection_name=COLLECTION_NAME,

        prefetch=[
            models.Prefetch(
                query=dense_query.tolist(),
                using=DENSE_VECTOR_NAME,
                limit=prefetch_limit,
            ),
            models.Prefetch(
                query=models.SparseVector(
                    indices=sparse_query.indices.tolist(),
                    values=sparse_query.values.tolist(),
                ),
                using=SPARSE_VECTOR_NAME,
                limit=prefetch_limit,
            ),
        ],

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        limit=top_k,
        with_payload=True,
    )

    rows = []

    for rank, point in enumerate(response.points, start=1):
        rows.append({
            "rank": rank,
            "score": point.score,
            "chunk_id": point.payload["chunk_id"],
            "story_id": point.payload["story_id"],
            "title": point.payload["title"],
            "text": point.payload["text"],
        })

    return pd.DataFrame(rows)


query = "испанская инквизиция"

retrieved = hybrid_retrieve(query, top_k=5)

display(retrieved[["rank", "score", "title", "text"]])

### 2.4 RAG

Собственно, RAG - это выполнение следующих шагов:
1. поиск релевантных чанков по запросу (гибридный поиск)
2. формирование контекста для генеративной модели на основе найденных n чанков
3. генерация ответа на основе сформированного контекста

In [ ]:
def build_context(chunks: pd.DataFrame, max_chars: int = 4500) -> str:
    parts = []

    for _, row in chunks.iterrows():
        part = (
            f"[Источник: {row['title']}, chunk_id={row['chunk_id']}]\n"
            f"{row['text']}"
        )
        parts.append(part)

    context = "\n\n---\n\n".join(parts)

    return context[:max_chars]


def ask_rag(question: str, top_k: int = 5) -> dict:
    retrieved = hybrid_retrieve(question, top_k=top_k)

    context = build_context(retrieved)

    messages = [
        {
            "role": "system",
            "content": (
                "Ты отвечаешь на вопросы по рассказам Ярослава Гашека. "
                "Отвечай только по предоставленному контексту. Не копируй текст."
                "Если ответа нет в контексте, скажи: "
                "'В предоставленном контексте нет ответа'. "
                "Отвечай кратко, но содержательно."
            )
        },
        {
            "role": "user",
            "content": f"""
Контекст:

{context}

Вопрос:
{question}

Ответ:
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    ).to(llm.device)

    with torch.no_grad():
        output_ids = llm.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
        )

    input_len = inputs["input_ids"].shape[-1]
    answer_ids = output_ids[0][input_len:]
    answer = tokenizer.decode(answer_ids, skip_special_tokens=True)

    return {
        "question": question,
        "answer": answer,
        "retrieved": retrieved,
        "context": context,
    }

In [ ]:
rag_questions = [
    "Что происходит в рассказе про озеро Балатон?",
    "Какой рассказ Гашека про испанскую инквизицию?",
    "В какой стране происходит действие рассказа про Швейка?"
]

for question in rag_questions:
    result = ask_rag(question, top_k=5)

    print("=" * 100)
    print("QUESTION:")
    print(result["question"])
    print()
    print("ANSWER:")
    print(result["answer"])
    # print("CONTEXT:")
    # print(result["context"])
    print()
    print("SOURCES:")
    display(result["retrieved"][["rank", "score", "title", "chunk_id"]])
    print("\n\n\n")

In [ ]:
rag_questions = [
    "Расскажи самую смешную историю про солдата Швейка?"
]

for question in rag_questions:
    result = ask_rag(question, top_k=5)

    print("=" * 100)
    print("QUESTION:")
    print(result["question"])
    print()
    print("ANSWER:")
    print(result["answer"])
    print()
    print("CONTEXT:")
    print(result["context"])
    # print("SOURCES:")
    # display(result["retrieved"][["rank", "score", "title", "chunk_id"]])

**ПРАКТИКА:**
1. Попробуйте поэкспериментировать с разными запросами и оценить качество результатов.
2. Поэкспериментируйте с чанкингом: попробуйте разные размеры чанков. Оцените качество результатов.
3. Можно ли передавать в генеративную модель не только текст чанков, но и другую информацию (например, метаданные)? Попробуйте поэкспериментировать с этим. Оцените как изменятся результаты.
4. Попробуйте поэкспериментировать с разными LLM, в том числе с облачными. !Убедитесь в том, что LLM ничего не знает о текстах, которые вы используете для RAG.

In [ ]:
# ваш код здесь



